In [ ]:
"""
CONTROLLING MODEL OUTPUT
-> It allows for more dynamic and interactive applications.

TWO TECHNIQUES:
-> prefilled assistant messages: Guide the model's behavior by providing example assistant messages in the conversation history.
-> stop sequences: Define specific sequences of text that, when generated by the model, will signal it to stop generating further content.
"""

# Load environment variables
from dotenv import load_dotenv

load_dotenv()

> **Note:** Claude Sonnet 5 and Claude Opus 5 (and the 4.6/4.7/4.8 family) no longer accept assistant message prefills — the API returns a 400 error. Older models like Claude Haiku 4.5 and Claude Sonnet 4.5 still support prefill, so this notebook uses `claude-haiku-4-5` to demonstrate the technique. On the newest flagship models, use structured outputs (`output_config.format`) or a system prompt instruction to control the response format instead.

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()

model = "claude-haiku-4-5"

In [ ]:
# Helpers
def add_message(messages, content, role):
    message = {"role": role, "content": content}
    messages.append(message)

"""
Stop Sequences
-> Force Claude to stop generating further content when it produces specific sequences of text.
-> Useful for controlling the length and format of the output.
"""

def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message.content[0].text

In [ ]:
"""
Message Prefilling
-> Provide the start of an assitant message as your last message
-> Claude will continue the response from there
-> This will greatly steer Claude's response
"""
messages = []

add_message(messages, 
            "Is tea or coffee better at breakfast?", 
            role="user")

# Prefill assistant message
add_message(messages, 
            # "Coffee is better because", 
            # "Tea is better because", 
            # "Both tea and coffee have their own unique benefits for breakfast. ",
            'Neither tea nor coffee is better because',
            role="assistant")

answer = chat(messages)

answer

In [ ]:
"""
Stop Sequences
"""
messages = []

add_message(messages, 
            "Count from 1 to 10", 
            role="user")

answer = chat(messages, stop_sequences=[", 5"])

answer

"""
Both prefilled messages and stop sequences give you fine-grained control over Claude's behavior, making them essential tools for 
building reliable AI applications.
"""



In [ ]:
"""
Structured data

-> Ask Clude to output data in a structured format like JSON, CSV, bullet points, etc.
"""

messages = []

add_message(messages, 
            "List 5 popular programming languages and their main use cases in JSON format.", 
            role="user")

# prefill assistant message
add_message(messages, "```json", role="assistant"),

# Stop sequence to end JSON block
answer = chat(messages, stop_sequences=["```"])

answer

In [ ]:
import json

data = json.loads(answer.strip())
data

# Structured Data Exercise

- Use message prefilling and stop sequence *only* to get three different commands in a single response
- There shouldn't be any comments or explanation
- Hint: message prefelling isn't limited to just chracters like ```

In [ ]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_message(messages, prompt, role="user")

text = chat(messages)
text.strip()

In [ ]:
from IPython.display import Markdown

Markdown(text)

In [ ]:
# SOLUTION
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""
add_message(messages, prompt, role="user")

# prefill assistant message
add_message(messages, "Here are three short AWS CLI commands:```bash", role="assistant"),

# Stop sequence to end JSON block
text = chat(messages, stop_sequences=["```"])

text.strip()

